# Quest Generation Pipeline Visualization

`agents/quest_generation/pipeline.py` 는 LangGraph 가 아니라 순수 async 루프다.
그래서 본 노트북은 두 갈래로 검증한다.

1. **architecture.mmd** 다이어그램을 그대로 렌더링 (코드와 다이어그램은 수동 동기화)
2. **FakeLLM 으로 파이프라인 실행** — 라운드-로빈, 쿼터 캡, LLM 실패 시 skipped 누적 동작을 직접 확인

구조 요약: `cap = min(len(todos), remaining_daily_quota)` → `CharacterPool` 라운드 셔플 → `LLMRunner.generate` (재시도 2회) → `generated` / `skipped` 분리.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.quest_generation import pipeline
from agents.quest_generation.exceptions import LLMFailedError
from agents.quest_generation.protocols import LLMPort
from agents.quest_generation.schemas import (
    Character,
    GeneratedQuest,
    QuestDistributionResult,
    QuestGenerationInput,
    SkippedItem,
    TodoRef,
)

print(pipeline.run.__module__, "·", pipeline.run.__qualname__)

agents.quest_generation.pipeline · run


## 1. Mermaid 소스 출력

`docs/features/quest_generation/architecture.mmd` 를 직접 읽어 출력한다.
https://mermaid.live 에 붙여넣으면 렌더링됨.


In [2]:
mmd_path = ROOT / "docs" / "features" / "quest_generation" / "architecture.mmd"
mermaid_src = mmd_path.read_text(encoding="utf-8")
print(mermaid_src)

---
config:
  layout: elk
---
flowchart TB
    A[/"Input
    QuestGenerationInput
    (todos, characters, remaining_daily_quota, shuffle_seed?)"/]

    A --> CAP["pipeline.run
    cap = min(len(todos), remaining_daily_quota)
    cap<=0 또는 characters 비어있음 → 즉시 빈 결과 반환"]

    CAP --> INIT["CharacterPool 생성
    seed로 결정성, 풀 비면 자동 셔플 리셋"]

    INIT --> LOOP["for todo in todos[:cap]"]

    LOOP --> SELECT["pool.next()
    풀 비면 _refill() → pop"]

    SELECT --> RUNNER{{"LLMRunner.generate
    LLMPort.generate_quest(character=...)
    재시도 max_retries=2 (총 3시도)"}}

    RUNNER -- 성공 --> ACC["generated 누적
    GeneratedQuest(character_id, todo_id, quest_text<=80자)"]

    RUNNER -- 모두 실패 --> SKIP["skipped 누적
    SkippedItem(todo_id, reason='llm_failure')"]

    ACC --> NEXT{"loop 끝?"}
    SKIP --> NEXT
    NEXT -- 아니오 --> LOOP
    NEXT -- 예 --> OUT[/"Output
    QuestDistributionResult(generated, skipped)"/]

    classDef input fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef process fill:#ff

## 2. Mermaid 인라인 렌더링

Mermaid.js CDN 을 사용해 노트북 안에서 직접 렌더링한다.
**인터넷 연결 필요.** JupyterLab / Jupyter Notebook 에서 동작.


In [3]:
from IPython.display import HTML, display

html = f"""
<div class="mermaid">
{mermaid_src}
</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{ startOnLoad: true }});
</script>
"""
display(HTML(html))

## 3. I/O 스키마 확인

Pydantic 모델 필드를 직접 출력 — `schemas.py` 변경 시 빠른 검증용.


In [4]:
def show(model):
    print(f"=== {model.__name__} ===")
    for name, field in model.model_fields.items():
        print(f"  - {name}: {field.annotation}")
    print()

for m in (TodoRef, Character, QuestGenerationInput, GeneratedQuest, SkippedItem, QuestDistributionResult):
    show(m)

=== TodoRef ===
  - todo_id: <class 'uuid.UUID'>

=== Character ===
  - character_id: <class 'uuid.UUID'>
  - name: <class 'str'>
  - personality: <class 'str'>
  - speech_style: <class 'str'>
  - appearance_keywords: list[str]

=== QuestGenerationInput ===
  - todos: list[agents.quest_generation.schemas.TodoRef]
  - characters: list[agents.quest_generation.schemas.Character]
  - remaining_daily_quota: <class 'int'>
  - shuffle_seed: int | None

=== GeneratedQuest ===
  - character_id: <class 'uuid.UUID'>
  - todo_id: <class 'uuid.UUID'>
  - quest_text: <class 'str'>

=== SkippedItem ===
  - todo_id: <class 'uuid.UUID'>
  - reason: typing.Literal['llm_failure']

=== QuestDistributionResult ===
  - generated: list[agents.quest_generation.schemas.GeneratedQuest]
  - skipped: list[agents.quest_generation.schemas.SkippedItem]



## 4. FakeLLM 정의 — 외부 호출 없이 파이프라인 검증

`LLMPort` 프로토콜만 만족하면 된다. 캐릭터 이름을 그대로 돌려주는 어댑터.


In [5]:
from uuid import uuid4


class EchoLLM:
    """성공 시나리오용: 캐릭터 이름을 포함한 짧은 퀘스트 문자열 반환."""

    async def generate_quest(self, *, character: Character) -> str:
        return f"{character.name} 가 오늘도 마을을 산책한다."


def make_chars(n: int) -> list[Character]:
    return [
        Character(
            character_id=uuid4(),
            name=f"C{i}",
            personality="호기심 많음",
            speech_style="부드러운 1인칭",
            appearance_keywords=["둥근눈"],
        )
        for i in range(n)
    ]


def make_todos(n: int) -> list[TodoRef]:
    return [TodoRef(todo_id=uuid4()) for _ in range(n)]


print("helper 준비 완료")

helper 준비 완료


## 5. 시나리오 A — 정상 흐름 (C1·C2·C3·C4 검증)

캐릭터 3명 × TODO 5건. `shuffle_seed=42` 로 결정적 순서.

기대: 5개 생성, 0개 skipped, 1라운드(3건) + 2라운드(2건).


In [6]:
chars = make_chars(3)
todos = make_todos(5)

result = await pipeline.run(
    QuestGenerationInput(
        todos=todos,
        characters=chars,
        remaining_daily_quota=10,
        shuffle_seed=42,
    ),
    ports=pipeline.Ports(llm=EchoLLM()),
)

print(f"generated: {len(result.generated)}  ·  skipped: {len(result.skipped)}\n")
id_to_name = {c.character_id: c.name for c in chars}
for i, g in enumerate(result.generated, 1):
    print(f"  {i}. char={id_to_name[g.character_id]}  ·  quest='{g.quest_text}'")

generated: 5  ·  skipped: 0

  1. char=C2  ·  quest='C2 가 오늘도 마을을 산책한다.'
  2. char=C0  ·  quest='C0 가 오늘도 마을을 산책한다.'
  3. char=C1  ·  quest='C1 가 오늘도 마을을 산책한다.'
  4. char=C0  ·  quest='C0 가 오늘도 마을을 산책한다.'
  5. char=C1  ·  quest='C1 가 오늘도 마을을 산책한다.'


In [7]:
# 라운드 의미 검증: 첫 3건은 캐릭터 중복 없음, 4·5번째에서 풀 리셋되어 새 라운드.
names = [id_to_name[g.character_id] for g in result.generated]
round1, round2 = set(names[:3]), set(names[3:])

print(f"round1 chars: {sorted(round1)}  → 3명 모두 등장? {round1 == {'C0', 'C1', 'C2'}}")
print(f"round2 chars: {sorted(round2)}  → 2명 (라운드 리셋 후 일부)")
print(f"generated 총 수: {len(result.generated)}  ==  min(5, 10) = 5 ? {len(result.generated) == 5}")

round1 chars: ['C0', 'C1', 'C2']  → 3명 모두 등장? True
round2 chars: ['C0', 'C1']  → 2명 (라운드 리셋 후 일부)
generated 총 수: 5  ==  min(5, 10) = 5 ? True


## 6. 시나리오 B — 일일 쿼터 캡 (C1)

TODO 5건이지만 `remaining_daily_quota=2`. 기대: 2건만 생성.


In [8]:
result_cap = await pipeline.run(
    QuestGenerationInput(
        todos=make_todos(5),
        characters=make_chars(3),
        remaining_daily_quota=2,
        shuffle_seed=42,
    ),
    ports=pipeline.Ports(llm=EchoLLM()),
)
print(f"generated={len(result_cap.generated)}  skipped={len(result_cap.skipped)}  → cap=2 적용? {len(result_cap.generated) == 2}")

generated=2  skipped=0  → cap=2 적용? True


## 7. 시나리오 C — 빈 입력 / 캐릭터 없음 / 쿼터 0

모두 예외 없이 빈 결과로 반환되어야 한다.


In [9]:
async def run_case(label, *, todos, characters, quota):
    r = await pipeline.run(
        QuestGenerationInput(
            todos=todos,
            characters=characters,
            remaining_daily_quota=quota,
            shuffle_seed=42,
        ),
        ports=pipeline.Ports(llm=EchoLLM()),
    )
    print(f"  {label:25s} → generated={len(r.generated)} skipped={len(r.skipped)}")


async def all_empty_cases():
    await run_case("todos=[]", todos=[], characters=make_chars(3), quota=10)
    await run_case("characters=[]", todos=make_todos(3), characters=[], quota=10)
    await run_case("quota=0", todos=make_todos(3), characters=make_chars(3), quota=0)


await all_empty_cases()

  todos=[]                  → generated=0 skipped=0
  characters=[]             → generated=0 skipped=0
  quota=0                   → generated=0 skipped=0


## 8. 시나리오 D — LLM 실패 시 skipped 누적

`FlakyLLM` 은 처음 N번 호출에서 `LLMFailedError` 를 던진다.
`LLMRunner.max_retries=2` 이므로 총 3시도까지 허용 → 4번째부터 실패면 skip.

구성: TODO 2건, 캐릭터 2명. 첫 번째 TODO 는 3시도 모두 실패 (skip), 두 번째 TODO 는 성공.


In [10]:
class FlakyLLM:
    def __init__(self, fail_first_n: int) -> None:
        self._left = fail_first_n
        self.calls = 0

    async def generate_quest(self, *, character: Character) -> str:
        self.calls += 1
        if self._left > 0:
            self._left -= 1
            raise LLMFailedError("forced failure")
        return f"{character.name} 가 햇살 아래 졸고 있다."


flaky = FlakyLLM(fail_first_n=3)  # 첫 TODO 의 3시도 모두 실패

result_fail = await pipeline.run(
    QuestGenerationInput(
        todos=make_todos(2),
        characters=make_chars(2),
        remaining_daily_quota=5,
        shuffle_seed=42,
    ),
    ports=pipeline.Ports(llm=flaky),
)

print(f"총 LLM 호출 수: {flaky.calls}  (기대: 첫 TODO 3시도 + 두 번째 TODO 1시도 = 4)")
print(f"generated: {len(result_fail.generated)}  skipped: {len(result_fail.skipped)}")
for s in result_fail.skipped:
    print(f"  skipped → todo_id={s.todo_id}  reason={s.reason}")

총 LLM 호출 수: 4  (기대: 첫 TODO 3시도 + 두 번째 TODO 1시도 = 4)
generated: 1  skipped: 1
  skipped → todo_id=024fc791-97d0-474f-bfe2-44f9ae9670c7  reason=llm_failure


## 9. (선택) 실제 Mi:dm 어댑터로 1건 실행

`adapters/quest_generation/midm_llm.py` 의 어댑터를 사용한 실주행.
환경변수 `MIDM_API_KEY` (+ 필요 시 `MIDM_BASE_URL`, `MIDM_MODEL`) 가 설정돼 있어야 한다.

키가 없으면 셀이 안내 메시지를 출력하고 넘어간다.


In [11]:
import os

if not os.getenv("MIDM_API_KEY"):
    print("MIDM_API_KEY 미설정 → 실주행 스킵.")
else:
    try:
        from adapters.quest_generation.midm_llm import MidmLLM  # type: ignore

        real_result = await pipeline.run(
                QuestGenerationInput(
                    todos=make_todos(1),
                    characters=make_chars(1),
                    remaining_daily_quota=1,
                    shuffle_seed=42,
                ),
                ports=pipeline.Ports(llm=MidmLLM()),
        )
        for g in real_result.generated:
            print(f"quest_text: {g.quest_text}")
        for s in real_result.skipped:
            print(f"skipped: {s.reason}")
    except Exception as exc:
        print(f"실주행 실패: {exc!r}")

MIDM_API_KEY 미설정 → 실주행 스킵.
